In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from medicalDataLoader_Enes import MedicalImageDataset, UnlabeledDataset, PseudoLabelDataset
from utils_Enes import device, DiceLoss, evaluate, make_pseudo_labels, show_predictions, show_confusion
from UNet_Base_Enes import UNet

In [ ]:
BASELINE_PATH = '../baseline/models/final_dice/best.pth'
MODEL_DIR = 'models/pseudo_labels'
EPOCHS = 50
BATCH_SIZE = 8
LR = 0.0001
SEED = 0
THRESHOLD = 0.95
PSEUDO_WEIGHT = 0.5
RAMPUP = 10

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../../Data/'

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

train_set = MedicalImageDataset('train', DATA_DIR, augment=True)
val_set = MedicalImageDataset('val', DATA_DIR)
unlabeled_set = UnlabeledDataset(DATA_DIR)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
unlabeled_loader = DataLoader(unlabeled_set, batch_size=BATCH_SIZE)

net = UNet(num_classes=4).to(device)
net.load_state_dict(torch.load(BASELINE_PATH, map_location=device))

In [ ]:
pseudo_masks = make_pseudo_labels(net, unlabeled_loader, THRESHOLD)
print('Pixels ignorés : {:.2f} %'.format(100 * (pseudo_masks == 255).mean()))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k, i in enumerate([0, 250, 500, 750]):
    axes[0, k].imshow(unlabeled_set[i][0], cmap='gray')
    axes[1, k].imshow(np.where(pseudo_masks[i] == 255, 4, pseudo_masks[i]), vmin=0, vmax=4)
plt.show()

pseudo_set = PseudoLabelDataset(DATA_DIR, pseudo_masks)
pseudo_loader = DataLoader(pseudo_set, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
ce_loss = nn.CrossEntropyLoss(ignore_index=255)
dice_loss = DiceLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

os.makedirs(MODEL_DIR, exist_ok=True)
history = {'train_loss': [], 'val_dice': [], 'val_hd': []}
best_dice = -1

for epoch in range(EPOCHS):
    net.train()
    weight = PSEUDO_WEIGHT * min(1, (epoch + 1) / RAMPUP)
    losses = []
    for (images, targets), (u_images, u_targets) in zip(train_loader, pseudo_loader):
        images, targets = images.to(device), targets.to(device)
        u_images, u_targets = u_images.to(device), u_targets.to(device)
        out = net(images)
        u_out = net(u_images)
        sup_loss = ce_loss(out, targets) + dice_loss(out, targets)
        pseudo_loss = ce_loss(u_out, u_targets) + dice_loss(u_out, u_targets)
        loss = sup_loss + weight * pseudo_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

    dice, hd = evaluate(net, val_loader)
    history['train_loss'].append(np.mean(losses))
    history['val_dice'].append(dice)
    history['val_hd'].append(hd)
    print('Epoch {} | poids {:.2f} | loss {:.4f} | Dice VD {:.3f} Myo {:.3f} VG {:.3f} | HD VD {:.1f} Myo {:.1f} VG {:.1f}'.format(
        epoch, weight, np.mean(losses), *dice, *hd))

    if np.mean(dice) > best_dice:
        best_dice = np.mean(dice)
        torch.save(net.state_dict(), MODEL_DIR + '/best.pth')

torch.save(net.state_dict(), MODEL_DIR + '/last.pth')
np.save(MODEL_DIR + '/history.npy', history)

In [ ]:
net.load_state_dict(torch.load(MODEL_DIR + '/best.pth', map_location=device))
dice, hd = evaluate(net, val_loader, post_process=True)
print('Dice VD {:.3f} Myo {:.3f} VG {:.3f} (moy. {:.3f}) | HD VD {:.1f} Myo {:.1f} VG {:.1f}'.format(*dice, np.mean(dice), *hd))

show_predictions(net, val_set, [5, 20, 40, 60], post_process=True)
show_confusion(net, val_loader, post_process=True)